libraries

In [7]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model

In [8]:
import numpy as np

X_train_scaled = np.load("X_train_scaled.npy")
X_test_scaled = np.load("X_test_scaled.npy")
y_train = np.load("y_train.npy")
y_test = np.load("y_test.npy")
X_train_normal = np.load("X_train_normal.npy")

print(X_train_scaled.shape)
print(X_train_normal.shape)

(22544, 116)
(9711, 116)


autoencoder base

In [9]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import tensorflow as tf

def build_autoencoder(input_dim, learning_rate=0.001, dropout_rate=0.2):
    """
    Builds an improved Autoencoder model for anomaly detection in network traffic.
    
    Improvements:
    - Added Dropout layers for regularization to reduce overfitting and false positives.
    - Added BatchNormalization for training stability and faster convergence.
    - Configurable learning rate with Adam optimizer.
    - Compiled with MSE loss and additional MAE metric for better reconstruction evaluation.
    - Wrapped in a function for modularity and reusability (e.g., for hyperparameter tuning).
    - Suggested callbacks for training (EarlyStopping and ReduceLROnPlateau) to prevent overfitting
      and improve efficiency, aligning with project goals of reliable detection and reduced false alarms.
    
    Args:
        input_dim (int): Number of input features (e.g., 41 for NSL-KDD dataset).
        learning_rate (float): Learning rate for the optimizer.
        dropout_rate (float): Dropout rate for regularization.
    
    Returns:
        Model: Compiled Keras Autoencoder model.
    """
    # Input layer
    input_layer = Input(shape=(input_dim,), name='input_features')
    
    # Encoder
    encoded = Dense(32, activation='relu', name='encoder_dense_1')(input_layer)
    encoded = BatchNormalization(name='encoder_bn_1')(encoded)
    encoded = Dropout(dropout_rate, name='encoder_dropout_1')(encoded)
    
    encoded = Dense(16, activation='relu', name='encoder_dense_2')(encoded)
    encoded = BatchNormalization(name='encoder_bn_2')(encoded)
    encoded = Dropout(dropout_rate, name='encoder_dropout_2')(encoded)
    
    latent = Dense(8, activation='relu', name='latent_space')(encoded)
    
    # Decoder (symmetric to encoder)
    decoded = Dense(16, activation='relu', name='decoder_dense_1')(latent)
    decoded = BatchNormalization(name='decoder_bn_1')(decoded)
    decoded = Dropout(dropout_rate, name='decoder_dropout_1')(decoded)
    
    decoded = Dense(32, activation='relu', name='decoder_dense_2')(decoded)
    decoded = BatchNormalization(name='decoder_bn_2')(decoded)
    decoded = Dropout(dropout_rate, name='decoder_dropout_2')(decoded)
    
    # Output layer for reconstruction
    output_layer = Dense(input_dim, activation='linear', name='reconstructed_features')(decoded)
    
    # Model definition
    autoencoder = Model(inputs=input_layer, outputs=output_layer, name='hybrid_nids_autoencoder')
    
    # Compile with improved optimizer and metrics
    optimizer = Adam(learning_rate=learning_rate)
    autoencoder.compile(
        optimizer=optimizer,
        loss='mse',
        metrics=['mae']  # Mean Absolute Error as additional reconstruction metric
    )
    
    return autoencoder

# Example usage (assuming X_train_normal is prepared from NSL-KDD or CICIDS2017)
# input_dim = X_train_normal.shape[1]  # e.g., 41 for NSL-KDD
# model = build_autoencoder(input_dim)

# Optional: Training callbacks for better performance (use during fit())
# callbacks = [
#     EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
#     ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
# ]

# # Train on normal traffic only
# history = model.fit(
#     X_train_normal, X_train_normal,
#     epochs=100,
#     batch_size=32,
#     validation_split=0.2,
#     callbacks=callbacks,
#     verbose=1
# )

# Display model summary
# model.summary()

In [10]:
input_dim = X_train_normal.shape[1]

autoencoder = build_autoencoder(input_dim)

autoencoder.summary()

Model: "hybrid_nids_autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_features (InputLayer)     │ (None, 116)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_dense_1 (Dense)         │ (None, 32)             │         3,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_bn_1                    │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_dropout_1 (Dropout)     │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_dense_2 (Dense)         │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_bn_2                    │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_dropout_2 (Dropout)     │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ latent_space (Dense)            │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_dense_1 (Dense)         │ (None, 16)             │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_bn_1                    │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_dropout_1 (Dropout)     │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_dense_2 (Dense)         │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_bn_2                    │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_dropout_2 (Dropout)     │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reconstructed_features (Dense)  │ (None, 116)            │         3,828 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,308 (36.36 KB)

 Trainable params: 9,116 (35.61 KB)

 Non-trainable params: 192 (768.00 B)